In [2]:
# @title ⚙️ Settings (chọn cặp model & cấu hình train) { display-mode: "form" }
# @markdown Chạy cell này trước. Có thể dùng dropdown của Colab để chọn `PAIR_KEY`.

REPO_URL = "https://github.com/duncan-nguyen/text_embedding_kd.git"  # @param {type:"string"}
REPO_DIR = "text_embedding_kd"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}

# 3 cặp model của bài
PAIR_KEY = "qwen3_0_6b_to_minilmv2_h384"  # @param ["qwen3_0_6b_to_minilmv2_h384", "bge_m3_to_minilmv2_h768", "qwen3_4b_to_bert_base"]

TRAIN_DATA = "data/train_set/merged_3_data_5k_each.csv"  # @param {type:"string"}
BATCH_SIZE = 128  # @param {type:"integer"}
EPOCHS = 5  # @param {type:"integer"}
LR = 2e-5  # @param {type:"number"}
MAX_LENGTH = 256  # @param {type:"integer"}
SEED = 42  # @param {type:"integer"}

SUBSPACE_RANK = 64  # @param {type:"integer"}
NUM_BLOCKS = 8  # @param {type:"integer"}
STABILITY_MARGIN = 0.05  # @param {type:"number"}
STABILITY_TAU = 0.05  # @param {type:"number"}
W_FUSION = 1.0  # @param {type:"number"}
TARGET_VIEW = "both"  # @param ["text1", "both"]
STABILITY_VIEW = "auto"  # @param ["auto", "dropout", "augment"]

INSTALL_DEPS = True  # @param {type:"boolean"}
USE_WANDB = False  # @param {type:"boolean"}
FORCE_RECOMPUTE = False  # @param {type:"boolean"}
USE_DRIVE = False  # @param {type:"boolean"}
DRIVE_DIR = "/content/drive/MyDrive/ourmethod"  # @param {type:"string"}

MODEL_PAIRS = {
    "qwen3_0_6b_to_minilmv2_h384": {
        "teacher": "Qwen/Qwen3-Embedding-0.6B",
        "student": "nreimers/MiniLMv2-L6-H384-distilled-from-BERT-Base",
        "pooling": "last_token",
        "dtype": "bfloat16",
    },
    "bge_m3_to_minilmv2_h768": {
        "teacher": "BAAI/bge-m3",
        "student": "nreimers/MiniLMv2-L6-H768-distilled-from-BERT-Base",
        "pooling": "cls",
        "dtype": "float32",
    },
    "qwen3_4b_to_bert_base": {
        "teacher": "Qwen/Qwen3-Embedding-4B",
        "student": "google-bert/bert-base-uncased",
        "pooling": "last_token",
        "dtype": "bfloat16",
    },
}

assert PAIR_KEY in MODEL_PAIRS, f"PAIR_KEY phải là một trong {list(MODEL_PAIRS)}"
PAIR = MODEL_PAIRS[PAIR_KEY]
print(f"Pair  : {PAIR_KEY}")
print(f"  teacher = {PAIR['teacher']}")
print(f"  student = {PAIR['student']}")
print(f"  pooling = {PAIR['pooling']}, dtype = {PAIR['dtype']}")

Pair  : qwen3_0_6b_to_minilmv2_h384
  teacher = Qwen/Qwen3-Embedding-0.6B
  student = nreimers/MiniLMv2-L6-H384-distilled-from-BERT-Base
  pooling = last_token, dtype = bfloat16


# Train OurMethod (student-anchored subspace fusion)

Notebook clone repo và train method `ourmethod`. Chọn 1 trong 3 cặp model
teacher–student đang dùng trong bài ở **cell Settings đầu tiên**.

Thứ tự: Settings → Clone/Pull → Install deps → (Drive) → Train → Results.

Lưu ý: rerun cell Clone sẽ tự `git fetch` + `git pull --ff-only` nếu repo đã tồn tại.
> Cần push code `ourmethod` lên remote trước khi chạy notebook (nếu remote chưa có).

## 1. Clone hoặc pull repo

Nếu `REPO_DIR/.git` đã tồn tại thì `git fetch` + `git pull --ff-only`; ngược lại thì clone mới.

In [3]:
import os
import subprocess
import sys
from pathlib import Path


def shell(cmd, cwd=None):
    print("$", " ".join(cmd))
    return subprocess.run(cmd, cwd=cwd, check=True)


if not REPO_URL or "github.com" not in REPO_URL:
    raise ValueError("\u0110\u1eb7t REPO_URL trong cell Settings tr\u01b0\u1edbc.")

repo_path = Path(REPO_DIR).expanduser().resolve()
if (repo_path / ".git").is_dir():
    print(f"Repo \u0111\u00e3 t\u1ed3n t\u1ea1i t\u1ea1i {repo_path} \u2192 fetch + pull...")
    shell(["git", "fetch", "origin"], cwd=repo_path)
    shell(["git", "checkout", BRANCH], cwd=repo_path)
    shell(["git", "pull", "--ff-only", "origin", BRANCH], cwd=repo_path)
else:
    print(f"Clone {REPO_URL} \u2192 {repo_path}")
    shell(["git", "clone", "--branch", BRANCH, REPO_URL, str(repo_path)])

os.chdir(repo_path)
print("Working dir:", Path.cwd())
shell(["git", "log", "--oneline", "-1"])

Clone https://github.com/duncan-nguyen/text_embedding_kd.git → /content/text_embedding_kd
$ git clone --branch main https://github.com/duncan-nguyen/text_embedding_kd.git /content/text_embedding_kd
Working dir: /content/text_embedding_kd
$ git log --oneline -1


CompletedProcess(args=['git', 'log', '--oneline', '-1'], returncode=0)

## 2. Cài dependencies

In [4]:
if INSTALL_DEPS:
    shell([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
else:
    print("B\u1ecf qua c\u00e0i dependencies (INSTALL_DEPS = False)")

$ /usr/bin/python3 -m pip install -q -r requirements.txt


## 3. (Tùy chọn) Mount Google Drive

Bật `USE_DRIVE=True` ở cell Settings nếu muốn lưu checkpoint/weights bền vững.

In [5]:
if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print("Drive mounted:", DRIVE_DIR)
else:
    print("USE_DRIVE = False")

USE_DRIVE = False


## 4. Train OurMethod

Lần đầu sẽ build fused target và cache vào
`cache/ourmethod/<PAIR_KEY>/targets*.pt`. Rerun chỉ load cache
(bật `FORCE_RECOMPUTE=True` để build lại).

In [6]:
import shlex

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

if USE_DRIVE:
    base_out = Path(DRIVE_DIR) / PAIR_KEY
    save_dir = str(base_out / "save")
    weights_dir = str(base_out / "weights")
else:
    save_dir = str(Path("models/ourmethod") / PAIR_KEY)
    weights_dir = str(Path("models/ourmethod_weights") / PAIR_KEY)

cmd = [
    sys.executable, "main.py",
    "--method", "ourmethod",
    "--train_data", TRAIN_DATA,
    "--student_model", PAIR["student"],
    "--base_student_model", PAIR["student"],
    "--teacher_model", PAIR["teacher"],
    "--teacher_pooling", PAIR["pooling"],
    "--teacher_dtype", PAIR["dtype"],
    "--cache_path", f"cache/ourmethod/{PAIR_KEY}/targets.pt",
    "--batch_size", str(BATCH_SIZE),
    "--epochs", str(EPOCHS),
    "--lr", str(LR),
    "--max_length", str(MAX_LENGTH),
    "--seed", str(SEED),
    "--subspace_rank", str(SUBSPACE_RANK),
    "--num_blocks", str(NUM_BLOCKS),
    "--stability_margin", str(STABILITY_MARGIN),
    "--stability_tau", str(STABILITY_TAU),
    "--w_fusion", str(W_FUSION),
    "--target_view", TARGET_VIEW,
    "--stability_view", STABILITY_VIEW,
    "--save_dir", save_dir,
    "--weights_dir", weights_dir,
]
if not USE_WANDB:
    cmd.append("--no_wandb")
if FORCE_RECOMPUTE:
    cmd.append("--force_recompute")

print(" ".join(shlex.quote(part) for part in cmd))
log_path = Path(save_dir) / "train_stdout.log"
log_path.parent.mkdir(parents=True, exist_ok=True)
quoted = " ".join(shlex.quote(part) for part in cmd)
# `tee` streams to the notebook (tqdm + traceback) and saves a copy on disk.
result = subprocess.run(
    f"{quoted} 2>&1 | tee {shlex.quote(str(log_path))}",
    shell=True,
)
if result.returncode != 0:
    print(f"\n\u274c Training failed (exit {result.returncode}). Traceback \u1edf tr\u00ean; log: {log_path}")
    raise SystemExit(result.returncode)
print("\u2705 Training completed")
print(f"Log: {log_path}")


/usr/bin/python3 main.py --method ourmethod --train_data data/train_set/merged_3_data_5k_each.csv --student_model nreimers/MiniLMv2-L6-H384-distilled-from-BERT-Base --base_student_model nreimers/MiniLMv2-L6-H384-distilled-from-BERT-Base --teacher_model Qwen/Qwen3-Embedding-0.6B --teacher_pooling last_token --teacher_dtype bfloat16 --cache_path cache/ourmethod/qwen3_0_6b_to_minilmv2_h384/targets.pt --batch_size 128 --epochs 5 --lr 2e-05 --max_length 256 --seed 42 --subspace_rank 64 --num_blocks 8 --stability_margin 0.05 --stability_tau 0.05 --w_fusion 1.0 --target_view both --stability_view auto --save_dir models/ourmethod/qwen3_0_6b_to_minilmv2_h384 --weights_dir models/ourmethod_weights/qwen3_0_6b_to_minilmv2_h384 --no_wandb


CalledProcessError: Command '['/usr/bin/python3', 'main.py', '--method', 'ourmethod', '--train_data', 'data/train_set/merged_3_data_5k_each.csv', '--student_model', 'nreimers/MiniLMv2-L6-H384-distilled-from-BERT-Base', '--base_student_model', 'nreimers/MiniLMv2-L6-H384-distilled-from-BERT-Base', '--teacher_model', 'Qwen/Qwen3-Embedding-0.6B', '--teacher_pooling', 'last_token', '--teacher_dtype', 'bfloat16', '--cache_path', 'cache/ourmethod/qwen3_0_6b_to_minilmv2_h384/targets.pt', '--batch_size', '128', '--epochs', '5', '--lr', '2e-05', '--max_length', '256', '--seed', '42', '--subspace_rank', '64', '--num_blocks', '8', '--stability_margin', '0.05', '--stability_tau', '0.05', '--w_fusion', '1.0', '--target_view', 'both', '--stability_view', 'auto', '--save_dir', 'models/ourmethod/qwen3_0_6b_to_minilmv2_h384', '--weights_dir', 'models/ourmethod_weights/qwen3_0_6b_to_minilmv2_h384', '--no_wandb']' returned non-zero exit status 1.

## 5. Kết quả & diagnostics

In các chỉ số offline của fused target và hiển thị 4 plot.

In [ ]:
import json

from IPython.display import Image, display

prefixes = ("alignment/", "subspace/", "stability/", "gate/", "target/")
metrics_path = Path(save_dir) / "metrics.jsonl"
if metrics_path.exists():
    for line in metrics_path.read_text().splitlines():
        record = json.loads(line)
        diagnostics = {k: v for k, v in record.items() if k.startswith(prefixes)}
        if diagnostics:
            print("OurMethod diagnostics:")
            for key in sorted(diagnostics):
                value = diagnostics[key]
                if isinstance(value, (int, float)):
                    print(f"  {key}: {value:.4f}")
                else:
                    print(f"  {key}: {value}")
            break

diagnostics_dir = Path(save_dir) / "diagnostics"
if diagnostics_dir.exists():
    for png in sorted(diagnostics_dir.glob("*.png")):
        print(png.name)
        display(Image(filename=str(png)))
else:
    print("Kh\u00f4ng t\u00ecm th\u1ea5y diagnostics trong", diagnostics_dir)